# Naive Bayes

## 1. Load the training data

In [1]:
from itertools import product
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.base import clone
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import CategoricalNB, GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, KBinsDiscretizer, OrdinalEncoder

# Find the data and preprocessing code from either notebook location.
working_dir = Path.cwd()
project_root = next(
    (path for path in (working_dir, *working_dir.parents)
     if (path / 'src' / 'preprocessing_pipeline.py').is_file()
     and (path / 'Dataset' / 'processed').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Run this notebook from the project root or a subfolder.')

sys.path.insert(0, str(project_root))
from src.preprocessing_pipeline import (
    TARGET, build_preprocessor, continuous_features, nominal_features,
    numerical_features, ordinal_features,
)

RANDOM_STATE = 42
BAD, GOOD = 0, 1
CLASS_NAMES = ['bad', 'good']
data_dir = project_root / 'Dataset' / 'processed'

In [2]:
X_train = pd.read_csv(data_dir / 'X_train.csv')
y_train = pd.read_csv(data_dir / 'y_train.csv')[TARGET]

print(f'Training set has {len(X_train)} rows and {X_train.shape[1]} features.')
display(y_train.value_counts().reindex([BAD, GOOD]).set_axis(CLASS_NAMES).to_frame('count'))

Training set has 700 rows and 21 features.


,count
bad,210
good,490


## 2. Tune and validate with stratified 5-fold cross-validation

Compare Gaussian Naive Bayes with a categorical version that groups numeric features into quantile bins and uses one code per categorical feature. Learn the bins and category codes inside each training fold.

Compare probability cutoffs using the same held-out rows in each fold. Select the highest mean bad-class F1, using recall and F1 variation to break ties.

### Set up Gaussian Naive Bayes

In [3]:
baseline = Pipeline([
    ('preprocessor', build_preprocessor()),
    ('model', GaussianNB(priors=[0.69, 0.31], var_smoothing=0.021)),
])

candidates = [('Gaussian', {}, baseline)]
smoothing_values = [0.001, 0.003, 0.01, 0.015, 0.021, 0.03, 0.05, 0.1, 0.2, 0.5, 1.0]
for smoothing in smoothing_values:
    params = {'var_smoothing': smoothing}
    model = clone(baseline).set_params(model=GaussianNB(**params))
    candidates.append(('Gaussian', params, model))

### Set up Categorical Naive Bayes

In [4]:
bin_counts = [3, 4, 5, 7]
alpha_values = [0.25, 0.5, 1.0, 2.0, 4.0]
for bins, alpha, include_ratio in product(bin_counts, alpha_values, [True, False]):
    numeric_columns = continuous_features if include_ratio else numerical_features
    # Shift the codes so an unseen category gets 0.
    categories = Pipeline([
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ('shift', FunctionTransformer(lambda values: values + 1)),
    ])
    preprocessor = ColumnTransformer([
        ('continuous', KBinsDiscretizer(n_bins=bins, encode='ordinal'), numeric_columns),
        ('categorical', categories, nominal_features + ordinal_features),
    ])
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('model', CategoricalNB(alpha=alpha)),
    ])
    params = {'bins': bins, 'alpha': alpha, 'include_amount_per_month': include_ratio}
    candidates.append(('Categorical', params, model))

### Get validation probabilities

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))
thresholds = np.arange(10, 91, 5) / 100

# The first probability column is for bad credit (label 0).
validation_probabilities = [
    cross_val_predict(candidate, X_train, y_train, cv=folds, method='predict_proba')[:, BAD]
    for _, _, candidate in candidates
]

### Compare probability cutoffs

In [6]:
# Reuse the same predictions for each cutoff instead of fitting again.
results = []
for index, (family, _, _) in enumerate(candidates):
    bad_probability = validation_probabilities[index]
    for threshold in thresholds:
        predictions = np.where(bad_probability >= threshold, BAD, GOOD)
        fold_f1 = [
            f1_score(
                y_train.iloc[rows], predictions[rows], pos_label=BAD, zero_division=0,
            )
            for _, rows in folds
        ]
        results.append({
            'Candidate': index,
            'Model': family,
            'Threshold': threshold,
            'Mean F1 (bad)': np.mean(fold_f1),
            'F1 SD': np.std(fold_f1),
            'Recall (bad)': recall_score(y_train, predictions, pos_label=BAD, zero_division=0),
        })

### Pick the best settings

In [7]:
baseline_result = next(
    result for result in results
    if result['Candidate'] == 0 and result['Threshold'] == 0.5
)
ranked_results = pd.DataFrame(results).sort_values(
    ['Mean F1 (bad)', 'Recall (bad)', 'F1 SD'],
    ascending=[False, False, True],
)
best_result = ranked_results.iloc[0]
_, best_params, selected_model = candidates[int(best_result['Candidate'])]
best_threshold = best_result['Threshold']

In [8]:
print(f'Evaluated {len(candidates)} model settings and {len(thresholds)} thresholds.')
display(pd.DataFrame(
    [baseline_result, best_result.to_dict()], index=['Previous', 'Selected'],
).drop(columns='Candidate').round(4))

Evaluated 52 model settings and 17 thresholds.


,Model,Threshold,Mean F1 (bad),F1 SD,Recall (bad)
Previous,Gaussian,0.5,0.6073,0.0291,0.8143
Selected,Categorical,0.2,0.6264,0.0261,0.8286


In [9]:
if best_params:
    display(pd.Series(best_params, name='Selected value', dtype=object).to_frame())
else:
    print('Using the previous model settings.')

,Selected value
bins,7
alpha,1.0
include_amount_per_month,True


### Fit on the full training set

In [10]:
best_model = clone(selected_model).fit(X_train, y_train)

## 3. Evaluate on the held-out test set

Compare the previous tuned model and the selected model on the same test rows. The model settings and cutoff come from cross-validation on the training set.

In [11]:
X_test = pd.read_csv(data_dir / 'X_test.csv')
y_test = pd.read_csv(data_dir / 'y_test.csv')[TARGET]

baseline_prediction = baseline.fit(X_train, y_train).predict(X_test)
# Find the probability column for bad credit before applying the cutoff.
bad_column = list(best_model.classes_).index(BAD)
bad_probability = best_model.predict_proba(X_test)[:, bad_column]
y_pred = np.where(bad_probability >= best_threshold, BAD, GOOD)

In [12]:
rows = []
for prediction in (baseline_prediction, y_pred):
    rows.append({
        'Accuracy': accuracy_score(y_test, prediction),
        'Precision (bad)': precision_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'Recall (bad)': recall_score(y_test, prediction, pos_label=BAD, zero_division=0),
        'F1 (bad)': f1_score(y_test, prediction, pos_label=BAD, zero_division=0),
    })
display(pd.DataFrame(rows, index=['Previous', 'Selected']).round(4))

,Accuracy,Precision (bad),Recall (bad),F1 (bad)
Previous,0.69,0.4885,0.7111,0.5792
Selected,0.70,0.5000,0.8111,0.6186


In [13]:
print('Confusion matrix for the selected model')
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=[BAD, GOOD]),
    index=pd.Index(CLASS_NAMES, name='Actual'),
    columns=pd.Index(CLASS_NAMES, name='Predicted'),
))

Confusion matrix for the selected model


Predicted,bad,good
Actual,,
bad,73,17
good,73,137


In [14]:
print(classification_report(
    y_test, y_pred, labels=[BAD, GOOD], target_names=CLASS_NAMES,
    digits=4, zero_division=0,
))

              precision    recall  f1-score   support

         bad     0.5000    0.8111    0.6186        90
        good     0.8896    0.6524    0.7527       210

    accuracy                         0.7000       300
   macro avg     0.6948    0.7317    0.6857       300
weighted avg     0.7727    0.7000    0.7125       300

